[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/drdave-teaching/OPIM5512-labs/blob/master/Module1/Week1_TechStack/Lab1_FirstCommit/notebooks/Lab1_Joint_EDA_and_Report.ipynb)

# Lab 1 - joint EDA and the report

## You cannot run this notebook until both pull requests are merged.

That is not a limitation. That is the lab.

Partner A produced `data/clean/weather_hourly.csv`. Partner B produced
`data/clean/demand_hourly.csv`. Neither of you has both files until `main` contains both -
which takes two branches, two pull requests, two reviews and two merges.

**Before you start:** in GitHub Desktop, switch to `main`, **Fetch origin**, then **Pull**.
Confirm you can see both files in `data/clean/`.

---

## How to work from here

**One screen, two people.** One of you drives (types), the other navigates (reads the plot,
asks the next question, catches the mistake). Swap after ten minutes.

Only the driver's laptop commits this notebook - remember, notebooks do not merge, so you
cannot both edit this file on separate branches. The navigator reviews the pull request. That
is pair programming, and it is how most real analysis actually gets done.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Point this at YOUR pair repo, on the main branch.
# github.com URL -> click Raw -> copy that URL -> keep everything up to /data/clean
REPO_RAW = "https://raw.githubusercontent.com/<your-org>/<your-repo>/main/data/clean"

weather = pd.read_csv(f"{REPO_RAW}/weather_hourly.csv", parse_dates=["hour"])
demand  = pd.read_csv(f"{REPO_RAW}/demand_hourly.csv",  parse_dates=["hour"])

df = demand.merge(weather, on="hour", how="inner")
print(len(df), "matched hours out of 744")
print("columns:", list(df.columns))   # do these match your contract?
df.head()

> ### Checkpoint
> **742-744** means your data contract held. Say it out loud - the two of you independently
> wrote code that agreed with each other. (You will not get exactly 744: the airport misses an
> observation now and then. Find out which hour, and put it in the report's caveats.)
>
> **0** means your timestamps disagree. Do not fix this by fiddling. Print both `hour` columns
> and find out which convention each of you chose. Then decide together which one is right and
> write it into the README. *That conversation is the actual skill.*
>
> **Around 700, or fewer** means something structural is wrong - a whole day dropped, or a
> timezone shift. Which side is short? Count the rows in each input before blaming the merge.

---

# 📋 The brief

> **From:** your manager
> **Subject:** weather and our load
>
> We keep hearing that hot weather drives up electricity demand. Before I take a position on
> it, I want to see it in our own data.
>
> Take a look and come back with a one-page summary. **I have about ninety seconds and I do
> not know what a correlation coefficient is.** Tell me what's going on, show me a picture or
> two, and be straight with me about what this data can't tell us.

That's your afternoon. You have **twenty-five minutes**.

## What you owe at the end

A file called `REPORT.md` in your repo, plus your figures in `figures/`. Copy the skeleton
from [REPORT_TEMPLATE.md](../REPORT_TEMPLATE.md) in the class repo to start.

**Three findings.** At least one has to be something that surprised you.

---

## Explore

There are no TODOs below this line. This is the part where you decide what to look at.

Some questions worth asking - **not a checklist, and not in any particular order.** Pick the
threads that look interesting and follow them:

- What does demand look like plotted against temperature? Is it a straight line?
- The grid has a daily rhythm. When does New England wake up, and when does it peak?
- Do weekends look different from weekdays?
- Which single hour of the month had the highest demand? Which was the hottest?
  Are they the same hour? The same *day*?
- Does humidity or dew point add anything beyond temperature, or is it just the same story?
- Pick the hottest day and plot that day's demand hour by hour. Now plot a mild day on the
  same axes. What's the difference, in MW?
- If you had to explain tomorrow's peak to your manager using **one number**, what would it be?

> 💡 **A number your manager will understand:** "each degree above 75 °F adds roughly ___ MW"
> lands far better than "r = 0.57". Try to produce one sentence of that shape.

### 🚀 A worked starting point

The join is done above — you've got `df` with both halves. Here's one thread carried through so
you're not staring at a blank cell: the **correlation**, the **manager sentence**
(*"each degree adds ~___ MW"*), and the **scatter** figure. Set `TEMP`/`LOAD` to your contract's
column names and run it.

This is a *running start, not the finish* — the questions above are still yours to chase, and the
last cell below points at the one finding most likely to surprise you.

In [ ]:
TEMP, LOAD = "tmpf", "load_mw"   # <-- set these to YOUR contract's column names
import numpy as np, os

# how tightly do temperature and demand move together?
r = df[TEMP].corr(df[LOAD])
print(f"correlation, temperature vs demand: r = {r:.2f}")

# the manager sentence: each degree of heat is worth how many MW?
slope, _ = np.polyfit(df[TEMP], df[LOAD], 1)
print(f"each +1 degF is worth about {slope:,.0f} MW of demand")

# the money figure -> saved for the report
os.makedirs("figures", exist_ok=True)
ax = df.plot.scatter(x=TEMP, y=LOAD, alpha=0.4, figsize=(8, 6),
                     title=f"Demand climbs with temperature (r = {r:.2f})")
ax.set_xlabel("temperature (degF)"); ax.set_ylabel("demand (MW)")
ax.get_figure().savefig("figures/demand_vs_temp.png", dpi=150, bbox_inches="tight")

In [ ]:
# TODO — worth chasing: is the HOTTEST hour also the PEAK-demand hour?
#   hottest = df.loc[df[TEMP].idxmax()]     # the single hottest hour
#   peak    = df.loc[df[LOAD].idxmax()]     # the single highest-demand hour
#   print both and compare — same hour? same day? how far apart in MW?
# If they're NOT the same, that's a real finding about your manager's question:
# heat matters, but the peak may be driven by time-of-day, not the thermometer.
# Write down what you conclude.

In [ ]:
# your exploration starts here


---

## Make the figures presentable

A figure going in front of a manager is not the same as a figure you make for yourself.
Before you export anything, check every one of these:

- Does it have a **title** that states the finding, not the variables?
  ("Demand climbs sharply above 75 °F" beats "load vs temp")
- Are both axes **labeled, with units**?
- Would someone who has never seen this data understand it without you talking?
- Have you removed everything that isn't carrying weight?

Two good figures beat six mediocre ones. Pick your best two or three.

In [ ]:
import os
os.makedirs("figures", exist_ok=True)

# Rebuild each keeper figure and save it. Use YOUR column names - the ones from
# your contract. Example shape:
#
# fig, ax = plt.subplots(figsize=(7, 5))
# ax.scatter(df[<your temperature column>], df[<your load column>], s=12, alpha=0.5)
# ax.set_xlabel("Temperature at the airport (deg F)")
# ax.set_ylabel("New England demand (MW)")
# ax.set_title("Demand climbs sharply above 75 degrees")
# fig.savefig("figures/demand_vs_temp.png", dpi=150, bbox_inches="tight")

print(os.listdir("figures"))

In [ ]:
# Zip them so it is one download instead of three.
import shutil
from google.colab import files

shutil.make_archive("figures", "zip", "figures")
files.download("figures.zip")

Unzip into your repo as `figures/`, then **GitHub Desktop** - new branch (call it
`report`), commit, push, open a PR. Your partner reviews it. That's the last loop of the
night.

---

## Write the report

Copy [REPORT_TEMPLATE.md](../REPORT_TEMPLATE.md) into your repo as `REPORT.md` and fill it in.
Both of you write - one drafts findings, the other drafts caveats, then swap and edit each
other's.

Three rules:

1. **Plain English.** If a sentence needs a statistics course to parse, rewrite it.
2. **Every number gets a unit.** MW, °F, hours. "Demand went up 40%" is weaker than
   "demand went up about 5,000 MW."
3. **Say what you don't know.** One month of summer data cannot tell you about January. A
   report that admits its limits is trusted; one that doesn't gets caught.

> 🔷 That last one is the difference between a student analysis and a professional one, and
> it's most of why anyone would keep asking you for work.

---

### Where this goes

That scatter plot is the seed of the semester. In **Module 2** you will fit a model to it and
argue about whether it generalizes. In **Module 3** you will make this data collect itself. In
**Module 4** you will try to forecast tomorrow's peak - and compare your forecast against the
one ISO-NE published, which is a genuinely humbling exercise.

And the caveat you just wrote about only having summer data? Module 4 hands you three years.
Go back and see whether the relationship you found still holds. It does not, entirely - and
finding out *how* it breaks is the interesting part.

Tonight you built the thing that makes all of that possible: a repo two people can work in
without breaking each other, and a habit of writing down what you found.